In [ ]:
# =============================================================================
# Spin-Wave Dispersion Relation — Calculation and Visualisation
# =============================================================================
# This notebook calculates and visualises the spin-wave dispersion relation
# from mumax3 micromagnetic simulation data. The workflow is:
#   1. Load time-domain magnetisation snapshots (.ovf files) from mumax3.
#   2. Apply a 2-D FFT over the time axis and one spatial axis to obtain the
#      frequency–wavevector (f–k) dispersion map.
#   3. Overlay the analytical Kalinikos-Slavin dispersion curve for comparison.
#   4. Optionally isolate individual spin-wave thickness modes and plot their
#      through-thickness amplitude profiles.
# =============================================================================

# --- Standard scientific libraries ---
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- mumax3 post-processing library (mumax3PP) ---
import mumax3PP.ovf as ovf                            # Reader for mumax3 .ovf output files
import mumax3PP.parameters as parameters              # Simulation parameter configuration
import mumax3PP.fft_across_xyzm as FFT_across_xyzm   # Multi-threaded FFT utilities

# --- Plotting utilities ---
import matplotlib.colors as colors       # Extended colormap support
from matplotlib import mlab, cm          # Colour map and MATLAB-style utilities

# --- Scientific computing ---
from scipy.optimize import curve_fit     # Non-linear least-squares fitting
from scipy.signal import argrelextrema  # Local extremum detection in 1-D arrays

# --- File system and I/O ---
import glob
from os import path
import time
from IPython.display import clear_output   # Clear notebook cell output during loops


%matplotlib inline

# --- Convenience aliases for common math functions ---
sqrt = np.sqrt
pi   = np.pi
exp  = np.exp
sin  = np.sin
cos  = np.cos
mu0  = np.pi * 4e-07   # Vacuum permeability (H/m)


In [2]:
# =============================================================================
# Cell 2: Helper functions and analytical dispersion relations
# =============================================================================

def findNearest(array, value):
    """
    Return the index and value of the element in `array` closest to `value`.

    Parameters
    ----------
    array : array-like  The array to search.
    value : float       The target value.

    Returns
    -------
    idx       : int    Index of the nearest element.
    array[idx]: float  The nearest value found.
    """
    idx = (np.abs(array - value)).argmin()
    return idx, array[idx]


def gedDT(times):
    """
    Robustly estimate the simulation time step dt from an array of output
    timestamps using a linear least-squares fit.

    A direct difference between adjacent samples can be noisy due to floating-
    point rounding in the mumax3 output. Fitting a line f(n) = dt*n + t0 over
    all timestamps gives a more stable estimate of dt.

    Parameters
    ----------
    times : array-like  Simulation timestamps (s).

    Returns
    -------
    float  Fitted time step dt (s) — the slope of the linear fit.
    """
    xdata = range(len(times))
    from scipy.optimize import curve_fit
    def func(x, a, b):
        """Linear model: a*x + b  (a = dt, b = t0)."""
        return a * x + b
    dt = np.abs(times[-1] - times[1]) / (len(times) - 1)   # Naive initial estimate
    popt, pcov = curve_fit(func, xdata, times, p0=[dt, 0])
    # Diagnostic plots (uncomment to verify the fit):
    #     plt.plot(xdata, times)
    #     plt.plot(xdata, func(xdata, *popt))
    return popt[0]   # Return the fitted slope = dt


def kalDR(k, angle_z, angle_k, d, mu0H0eff, wM, lb, mu0Ha, gamma):
    """
    Kalinikos-Slavin spin-wave dispersion relation for a thin magnetic film.

    Computes the spin-wave frequency at wavevector k using the analytical
    Kalinikos-Slavin formalism, which accounts for exchange, dipolar, and
    Zeeman contributions. Valid for the fundamental thickness mode.

    Parameters
    ----------
    k         : float or array  Wavevector magnitude (rad/m).
    angle_z   : float           Polar angle of equilibrium magnetisation
                                w.r.t. the film normal (rad). pi/2 = in-plane.
    angle_k   : float           Azimuthal angle of k w.r.t. the applied field
                                direction (rad). 0 = Backward Volume, pi/2 = Damon-Eshbach.
    d         : float           Film thickness (m).
    mu0H0eff  : float           Effective applied field, mu0 * Heff (T).
                                Includes demagnetisation correction.
    wM        : float           Characteristic angular frequency, gamma*mu0*Ms (rad/s).
    lb        : float           Exchange stiffness parameter, mu0*Ms*lex^2 (m^2·rad/s).
    mu0Ha     : float           Uniaxial anisotropy field, mu0*Ha (T). Zero for Permalloy.
    gamma     : float           Gyromagnetic ratio (rad/(s·T)).

    Returns
    -------
    float or array
        Spin-wave frequency in Hz.

    Notes
    -----
    The anisotropy term (gamma*mu0Ha) is currently excluded from w0 (commented out).
    The dipolar matrix element F is computed via the nested function `Fcoef1`.
    """
    def Fcoef1(k, theta, phi, w0):
        """
        Dipolar matrix element F(k) in the Kalinikos-Slavin formalism.

        Accounts for the finite film thickness through the pinning factor P,
        which interpolates between the magnetostatic surface and volume limits.

        Parameters
        ----------
        k   : wavevector magnitude (rad/m)
        theta : polar angle of magnetisation w.r.t. film normal (rad)
        phi   : azimuthal angle of k w.r.t. field (rad)
        w0  : FMR angular frequency including exchange (rad/s)
        """
        # Dipolar pinning factor P: ranges from 0 (surface wave) to 1 (volume wave)
        P = 1 - (1 - exp(-k * d)) / (k * d)
        return (P + (sin(theta)**2) * (1 - P * (1 + cos(phi)**2)
                + (sin(phi)**2) * (wM / w0) * (1 - exp(-2 * k * d)) / 4))

    # FMR frequency: Zeeman + exchange stiffness terms (anisotropy term commented out)
    w0 = gamma * mu0H0eff + gamma * lb * k**2   # + gamma*mu0Ha

    # Squared angular frequency from the Kalinikos-Slavin secular equation
    w2 = w0 * (w0 + wM * Fcoef1(k, angle_z, angle_k, w0))

    return np.sqrt(w2) / (2 * np.pi)   # Convert angular frequency to Hz


def fqKal_n(k_x, N, L_z, W_y, phi_z, phi_H, mu0Heff, wM, lb, mu0Ha, gamma):
    """
    Kalinikos-Slavin dispersion for the N-th quantised width mode (standing wave
    across the waveguide width W_y).

    In a finite-width magnetic strip, the transverse wavevector ky is quantised
    by the boundary conditions at the strip edges. This function computes the
    effective quantised ky for mode N and then evaluates the full 2-D dispersion
    at the resulting total wavevector k = sqrt(kx^2 + ky_N^2).

    Parameters
    ----------
    k_x     : float or array  Longitudinal wavevector kx (rad/m).
    N       : int             Width-mode index (0 = fundamental, 1, 2, … = higher modes).
    L_z     : float           Film thickness (m).
    W_y     : float           Strip width (m).
    phi_z   : float           Polar angle of magnetisation (rad).
    phi_H   : float           Azimuthal angle of applied field (rad).
    mu0Heff : float           Effective field mu0*Heff (T).
    wM      : float           Characteristic angular frequency gamma*mu0*Ms (rad/s).
    lb      : float           Exchange parameter (m^2·rad/s).
    mu0Ha   : float           Anisotropy field mu0*Ha (T).
    gamma   : float           Gyromagnetic ratio (rad/(s·T)).

    Returns
    -------
    float or array  Spin-wave frequency for mode N at wavevector kx (Hz).

    Notes
    -----
    Two quantisation models are defined for ky_N:
      - Free boundary conditions: ky_N = N*pi/W_y  (first lambda, currently inactive)
      - Guslienko pinning model:  ky_N = (N+1)*pi/W_y * (1 - 2/d)  (active)
    The Guslienko model accounts for surface-spin pinning via the factor d(W_y, L_z),
    which depends on the aspect ratio of the strip cross-section.
    """
    # Effective dipolar pinning length scale (Guslienko model)
    # d depends on the aspect ratio p = L_z / W_y and corrects ky for surface pinning
    d = lambda W_y, L_z: 2 * pi / (L_z / W_y * (1 + 2 * np.log(W_y / L_z)))

    # Free boundary conditions (inactive — commented out):
    # r = N   →   ky_N = N*pi/W_y
    r = N
    ky_n_free = lambda N, W_y, L_z: r * pi / W_y   # Free BC: n = 0, 1, 2, 3

    # Guslienko model with surface pinning (active):
    # ky_N = (N+1)*pi/W_y * (1 - 2/d)  reduces ky relative to the free-BC value
    ky_n = lambda N, W_y, L_z: (N + 1) * pi / W_y * (1 - 2 / d(W_y, L_z))

    k_y = ky_n(N, W_y, L_z)                 # Quantised transverse wavevector (rad/m)
    k_n = np.sqrt(k_x**2 + k_y**2)          # Total wavevector magnitude (rad/m)

    # Azimuthal angle of the total k vector w.r.t. the applied field
    phi = phi_H + np.arctan(k_y / k_x)

    # Evaluate the Kalinikos-Slavin dispersion at (k_n, phi)
    f_n = kalDR(k_n, phi_z, phi, L_z, mu0Heff, wM, lb, mu0Ha, gamma)
    return f_n


In [3]:
# =============================================================================
# Cell 3: Temporal resolution and frequency axis parameters
# =============================================================================
# These parameters define the time-domain sampling of the simulation output
# and directly determine the resolution and range of the frequency axis in
# the dispersion map.

f_cut     = 45e9    # Target cut-off frequency (Hz); used to guide t_sampl choice
t_sampl   = 5e-12   # Simulation output time step, dt (s); 5 ps here
                    # Rule of thumb: t_sampl < 1/(2*f_max) to satisfy Nyquist
                    # Commented alternative: t_sampl = 0.5/(f_cut*1.5)
n_samples = 800     # Number of time snapshots used for the FFT

# Derived frequency-axis quantities
df    = 1 / (t_sampl * n_samples)   # Frequency resolution (Hz): df = 1/T_total
f_max = 1 / (2 * t_sampl)           # Nyquist frequency (Hz): maximum resolvable frequency

print("Max frequency: {} GHz".format(f_max * 1e-9))     # Should exceed the highest mode of interest
print("time sampling: {} ps".format(t_sampl * 1e12))
print("freq sampling: {} GHz".format(df * 1e-9))        # Frequency bin width in the FFT output


In [4]:
# =============================================================================
# Cell 4: Spatial resolution and wavevector axis parameters
# =============================================================================
# These parameters define the spatial sampling of the simulation grid and
# set the resolution and range of the wavevector (k) axis in the dispersion map.

y_sampl   = 5e-9    # Spatial cell size along y (m); 5 nm here
n_samples = 4096    # Number of spatial grid points along y used for the FFT

# Derived wavevector-axis quantities
dk_cal = 2 * np.pi / (y_sampl * n_samples)   # Wavevector resolution (rad/m): dk = 2pi/L_total
k_max  = 2 * np.pi / (2 * y_sampl)           # Nyquist wavevector (rad/m):   k_max = pi/dy

print("Max k: {} rad/mum".format(k_max * 1e-6))      # Maximum resolvable wavevector
print("k sampling: {} rad/mum".format(dk_cal * 1e-6))  # Wavevector bin width


In [5]:
# Path to the mumax3 simulation output directory containing .ovf snapshots.
# This simulation models a 1024-cell Damon-Eshbach geometry with
# 0.5 nm spatial resolution in the cross-section (cz = 1 nm thickness).
dir0 = r"D:\mumax3\inelastic_scattering\dispersion\dispersion_1024_05nm_cz_1nm_DE.out"


In [6]:
# =============================================================================
# Cell 7: Load magnetisation data and build spatial coordinate arrays
# =============================================================================

tStop = -1   # Load all available time steps (-1 = no cutoff)

# Configure the OVF reader: load the magnetisation vector field ('m')
parms    = parameters.ovfParms(head="m", tStop=tStop)
M_tzyxm  = ovf.OvfFile(dir0, parms)

print((M_tzyxm.array).shape)   # Expected: (N_t, N_z, N_y, N_x, 3)

# Full array slice (all time steps and spatial points)
# Shape: (N_t, N_z, N_y, N_x, 3) — time × z-layers × y-cells × x-cells × (mx, my, mz)
M1 = M_tzyxm.array[:, :, :, :, :]
t1 = M_tzyxm.time   # Simulation timestamps (s)

# Estimate the time step from the timestamp array using a linear fit
dt = gedDT(M_tzyxm.time)

# --- Build real-space coordinate arrays from simulation header metadata ---
# Each axis is constructed as a uniform grid of cell-centre positions.
# The array is then shifted so that the origin (0) sits at the geometric centre.

cx = M_tzyxm._headers["xstepsize"]   # Cell size along x (m)
x  = np.arange(0, M1.shape[3] * cx - cx / 10, cx)   # x positions (m)
x -= x.max() / 2                                      # Centre x at 0

cy = M_tzyxm._headers["ystepsize"]   # Cell size along y (m)
y  = np.arange(0, M1.shape[2] * cy - cy / 10, cy)   # y positions (m)
y -= y.max() / 2                                      # Centre y at 0

cz = M_tzyxm._headers["zstepsize"]   # Cell size along z — film thickness (m)
z  = np.arange(0, M1.shape[1] * cz - cz / 10, cz)   # z positions (m)
z -= z.max() / 2                                      # Centre z at 0

# --- External field data (commented out; uncomment if B_ext analysis is needed) ---
# parms    = parameters.ovfParms(head="B_ext", tStop=tStop)
# B_tzyxm  = ovf.OvfFile(dir0, parms)
# print((B_tzyxm.array).shape)
# B_ext = B_tzyxm.array[:, :, :, :, :]


In [7]:
# =============================================================================
# Cell 9: Material parameters and analytical dispersion curve
# =============================================================================

# --- Permalloy (Ni80Fe20) material parameters ---
mu0   = pi * 4e-07   # Vacuum permeability (H/m)
d     = 10e-09       # Film thickness (m)
Ms    = 800e03       # Saturation magnetisation (A/m)
A     = 13e-12       # Exchange stiffness constant (J/m)
gamma = 175.95e9     # Gyromagnetic ratio (rad/(s·T))

# --- Derived magnetic parameters ---
wM    = gamma * mu0 * Ms         # Angular frequency scale wM = gamma*mu0*Ms (rad/s)
Ku    = 0                        # Uniaxial anisotropy constant (J/m^3); zero for Py
mu0Ha = 2 * Ku / Ms              # Anisotropy field mu0*Ha (T); zero here
lex2  = 2 * A / (mu0 * Ms**2)   # Squared exchange length lex^2 (m^2)
lb    = mu0 * Ms * lex2          # Exchange parameter lb = mu0*Ms*lex^2 (m^2·rad/s)

# --- Applied field ---
B0       = 0.3        # Applied field mu0*H0 (T)
mu0Heff  = B0         # Effective field = applied field (demagnetisation neglected here)

print("exchange length, l_ex= " + str(round(np.sqrt(lex2) * 1e9, 2)) + " (nm)")

# --- Evaluate the analytical dispersion along a 1-D k array ---
# Wavevector range: near-zero to 300 rad/µm, covering the accessible k-window
k = np.linspace(-0.1e6, 300e6, int(1e03))   # k values (rad/m)

# Geometry: Damon-Eshbach configuration
# angle_k = 90° - 30° = 60°: k is at 60° to the applied field (near DE geometry)
# angle_z = 90°: magnetisation lies in-plane (no canting)
angle_k = (90 - 30) * (np.pi / 180)   # Azimuthal angle of k w.r.t. H (rad)
angle_z = 90 * (np.pi / 180)          # Polar angle of M w.r.t. film normal (rad)

# Compute the Kalinikos-Slavin dispersion; np.abs(k) ensures the function
# receives positive-definite wavevector values (dispersion is symmetric in k)
fq = kalDR(np.abs(k), angle_z, angle_k, d, mu0Heff, wM, lb, mu0Ha, gamma)

# Diagnostic plot of the 1-D dispersion (commented out; shown on the full map later)
# plt.plot(k*1e-6, fq*1e-9, c="lime")
# plt.show()


In [8]:
# =============================================================================
# Cell 11: Compute the 2-D FFT dispersion map (f–ky)
# =============================================================================
# Strategy:
#   1. Reduce the 5-D magnetisation array to a 2-D (time × y) matrix by summing
#      over the x and z spatial dimensions. This integrates out the cross-section
#      and gives a signal representative of all spin waves propagating along y.
#   2. Apply a 2-D FFT simultaneously along the time axis (→ frequency) and the
#      y spatial axis (→ wavevector ky).
#   3. Retain only positive frequencies (physical spectrum) and shift the
#      wavevector axis so that ky = 0 is at the array centre.

# Sum over z-layers (axis 1) and x-cells (axis 3) to get a (N_t, N_y) array.
# Using component index 1 (my) as the signal; other components can be substituted.
m_tx = np.sum(M1[:, :, :, :, 1], axis=(1, 3))   # Shape: (N_t, N_y)
# Alternative: single cell extraction instead of summing:
# m_tx = M1[:, -1, :, 64, 0]   # Last z-layer, central x-column, mx component

# 2-D FFT: axis 0 = time → frequency, axis 1 = y-position → ky
m_fk = np.fft.fft2(m_tx, axes=(0, 1))   # Shape: (N_t, N_y), complex

sh = m_fk.shape

# Keep only the first half of the time-FFT output (positive frequencies)
# and fftshift along the spatial axis (axis 1) to centre ky = 0.
DR_i = np.fft.fftshift(m_fk[:int(sh[0] / 2), :], axes=(1))   # Shape: (N_t/2, N_y)

# --- Build frequency and wavevector axes ---
# fftfreq returns frequencies in cycles/sample; multiply by 2*pi for rad/m (wavevector)
kLi = np.fft.fftfreq(sh[1], cx) * 2 * pi   # Wavevector axis (rad/m); centred after fftshift
fLi = np.fft.fftfreq(sh[0], dt)             # Frequency axis (Hz); only positive half used

# Useful axis metadata for building plot extents
kMax, dk = kLi.max(), kLi[1]   # Maximum k and bin spacing (rad/m)
fMax, df = fLi.max(), fLi[1]   # Maximum frequency and bin spacing (Hz)


In [9]:
# =============================================================================
# Cell 12: Plot the experimental dispersion map and analytical overlay
# =============================================================================

# Extract the simulation name from the directory path for labelling
file_name = dir0.split("\\")[-1][:-4]

plt.rcParams.update({'font.size': 16})

plt.figure(figsize=(10, 10))
# plt.title(file_name)   # Uncomment to add the simulation name as a title

# --- Plot the 2-D FFT amplitude as the dispersion map ---
# The colour scale is strongly clipped (vmax = 0.5% of the peak) to reveal
# the weaker higher-order modes that would be hidden behind the dominant mode.
im = plt.imshow(
    np.abs(DR_i),
    extent=[(-kMax - dk / 2) * 1e-06, (kMax - dk / 2) * 1e-06,   # k axis in rad/µm
            0, (fMax - df) * 1e-09],                               # f axis in GHz
    vmin=0, vmax=0.005 * np.abs(DR_i).max(),   # Clip to 0.5% of peak to show weak modes
    aspect="auto", cmap="inferno", origin="lower",
    # interpolation="gaussian",   # Uncomment for smoother rendering
)

# --- Compute and optionally overlay the analytical Kalinikos-Slavin dispersion ---
k  = np.linspace(-300e6, 300e6, int(1e03))   # Full symmetric k range (rad/m)
fq = kalDR(np.abs(k), angle_z, angle_k, d, mu0Heff, wM, lb, mu0Ha, gamma)

# FMR frequency: evaluated at near-zero k (k → 0 limit of the dispersion)
f_FMR = kalDR(1e-6, angle_z, angle_k, d, mu0Heff, wM, lb, mu0Ha, gamma)
print(fq.min())   # Print the minimum frequency in the dispersion (should match f_FMR)

# Analytical dispersion overlay (commented out; uncomment to compare with simulation)
# plt.plot(k * 1e-6, fq * 1e-9, c="lime")

# Width-mode dispersions (N = 1..4) for finite-width strip geometry
# (commented out; uncomment for waveguide simulations with quantised ky)
# for N in range(1, 5):
#     fq1 = fqKal_n(np.abs(k), N, d, d, angle_z, angle_k, mu0Heff, wM, lb, mu0Ha, gamma)
#     print(fq1.min())
#     plt.plot(k * 1e-06, fq1 * 1e-09)

# --- Horizontal reference lines marking frequency regions of interest ---
plt.axhline(10.5)   # Lower bound of the frequency window of interest (GHz)
plt.axhline(17)     # Upper bound (GHz)
# plt.axvline(k0 * 1e-6)   # Uncomment to mark a specific wavevector

plt.xlim(-300, 300)   # Wavevector display range (rad/µm)
plt.ylim(0, 100)      # Frequency display range (GHz)

plt.xlabel(r"wavevector, $k_\mathrm{y}$ (rad/$\mathrm{\mu}$m)")
plt.ylabel(r"frequency, $f$ (GHz)")

# plt.colorbar(im, shrink=0.3)   # Uncomment to add a colour bar
# plt.savefig(...)               # Uncomment to save the figure
plt.show()


In [10]:
# Display the FMR frequency in GHz — the zero-k intercept of the dispersion curve.
# This is the minimum spin-wave frequency for this field and geometry.
f_FMR * 1e-9


In [ ]:
### Simple visualization
# =============================================================================
# The cells below isolate a single spin-wave thickness mode at a chosen
# frequency and wavevector, and plot its through-thickness (z) amplitude profile.
# This reveals whether the mode is a bulk (uniform through z) or a surface mode
# (amplitude concentrated near one film surface).
# =============================================================================


In [11]:
# =============================================================================
# Cell 17: Extract a specific spin-wave mode for through-thickness profiling
# =============================================================================
# The full 5-D data is reduced by fixing the x-column index and computing a
# 2-D FFT over time and y simultaneously, yielding a (frequency × z-layer × ky)
# cube. A single (f, ky) point can then be selected to isolate one mode.

comp = 2    # Magnetisation component to analyse: 0=mx, 1=my, 2=mz
x0   = 64   # Fixed x-column index (mid-width of the simulation cell)

# Extract m(t, z, y) at fixed x-column x0 and magnetisation component comp.
# Array axes: (t, z, y, x, m) → select x=x0 and component=comp → shape (N_t, N_z, N_y)
m_tzyx = M1[:, :, :, x0, comp]   # Shape: (N_t, N_z, N_y)

# 2-D FFT over time (axis 0) and y-position (axis 2):
#   axis 0: t → frequency f
#   axis 2: y → wavevector ky
# The z axis (axis 1) is preserved, giving the through-thickness information.
m_fzkyx = np.fft.fft2(m_tzyx, axes=(0, 2))   # Shape: (N_f, N_z, N_ky)

sh = m_fzkyx.shape

# Build the wavevector and frequency axes for this extraction
kLi = np.fft.fftfreq(sh[2], cy) * 2 * pi   # ky axis (rad/m)
fLi = np.fft.fftfreq(sh[0], dt)             # Frequency axis (Hz)


In [12]:
# =============================================================================
# Cell 18: Select target (f, k) point, apply k-filter, and inverse-FFT
# =============================================================================
# To isolate the mode at a specific (frequency, wavevector) point:
#   1. Find the indices in fLi and kLi closest to the target values.
#   2. Zero out the negative-k half of the spectrum (keep only the
#      forward-propagating component).
#   3. Apply a 1-D inverse FFT along ky to recover the real-space
#      spatial amplitude profile as a function of (z, x).

# --- Target wavevector ---
k_sample = 250e6                           # Target ky (rad/m)
k_ind    = findNearest(kLi, k_sample)[0]   # Nearest index in kLi

# --- Target frequency ---
# f_sample = f_FMR   # Uncomment to use the FMR frequency automatically
f_sample = 43e9      # Target frequency (Hz); manually set to 43 GHz here
f_ind    = findNearest(fLi, f_sample)[0]   # Nearest index in fLi

# Select the 2-D slice at the target frequency: shape (N_z, N_ky)
sample = m_fzkyx[f_ind, :, :]

# --- Zeroing negative-k components ---
# FFT output is ordered [0, +k, ..., -k, ...]; the first N/2 entries are
# positive-k (forward wave), the second half are negative-k (backward wave).
# Setting the negative-k half to zero isolates the forward-propagating mode.
sample_zeros = np.zeros_like(sample, dtype=complex)
k0 = int(sample.shape[1] / 2)       # Index boundary between +k and -k halves
sample[:, :k0] = sample_zeros[:, :k0]   # Zero out negative-k half

# 1-D inverse FFT along the ky axis (axis 1) to recover the real-space
# amplitude profile as a function of z (rows) and x (columns)
wave = np.fft.ifft(sample, axis=1)   # Shape: (N_z, N_x)


In [13]:
# =============================================================================
# Cell 19: Plot the 2-D real-space amplitude map (z vs. x) of the isolated mode
# =============================================================================
# This image shows how the spin-wave amplitude is distributed across the film
# cross-section (z-axis) and along the propagation direction (x-axis).

colormap = np.abs(wave)   # Amplitude envelope of the filtered mode

plt.figure(figsize=(10, 5))
plt.imshow(
    colormap,
    extent=[x.min() * 1e6, x.max() * 1e6,    # x axis in µm
            z.min() * 1e9, z.max() * 1e9],    # z axis in nm
    vmin=0, vmax=0.99 * colormap.max(),        # Clip top 1% to improve contrast
    aspect="auto", cmap="inferno", origin="lower", interpolation="none",
)
plt.show()


In [14]:
# Sanity check: print the maximum z coordinate in nm to verify the film thickness
z.max() * 1e9


In [15]:
# =============================================================================
# Cell 21: Plot the through-thickness (z) amplitude profile of the isolated mode
# =============================================================================
# A vertical cut through the 2-D amplitude map at a chosen x-column reveals
# the mode profile across the film thickness. A uniform (bulk) mode has a flat
# profile; a surface mode is concentrated near one face of the film.

# --- Label the magnetisation component for the plot title ---
if comp == 1:
    mag_comp = r"$m_x$"   # x-component
elif comp == 2:
    mag_comp = r"$m_z$"   # z-component (used here)

plt.rcParams.update({'font.size': 9})
plt.figure(figsize=(6, 4))
plt.title("bulk mode: {}; frequency, f = {} GHz".format(
    mag_comp, round(fLi[f_ind] * 1e-9)))

# --- Extract and plot the amplitude profile at each selected y-column ---
# y_Li lists the y-index positions at which to extract the cut-line.
# Multiple values can be used to check spatial uniformity along y.
y_Li = [600]   # Single y-column near the centre of the strip
# y_Li = np.arange(y.shape[0]//2 + 50, y.shape[0], 100)   # Range of columns

for y0 in y_Li:
    cutline = np.abs(wave[:, y0])     # 1-D amplitude profile along z at this y-column
    cutline /= cutline.max()          # Normalise to 1 for shape comparison across modes
    color = "tab:blue"
    plt.plot(cutline, z * 1e9, lw=3, c=color)                        # Profile curve
    plt.fill_betweenx(z * 1e9, 0, cutline, color=color, alpha=0.1)  # Shaded area

plt.xlim(0, 1.05)   # Normalised amplitude axis

# --- Custom z-axis ticks: label film boundaries and centre ---
y_ticks     = np.array([-5, 0, 5])                              # Labels in nm
y_ticks_pos = np.array([z.min() * 1e9, 0, z.max() * 1e9])      # Positions in nm
plt.yticks(y_ticks_pos, y_ticks)

# Dashed lines marking the top and bottom film surfaces
lw = 0.5
plt.axhline(y=z.max() * 1e9, ls="--", lw=lw, c="k")    # Top surface
plt.axhline(y=z.min() * 1e9, ls="--", lw=lw, c="k")    # Bottom surface
plt.axvline(x=1,              ls="--", lw=lw, c="magenta")  # Normalised amplitude = 1

plt.xlabel(r"|{}| (normalized)".format(mag_comp))
plt.ylabel(r"z (nm)")

# Save the figure at high resolution (600 dpi) for publication
plt.savefig(r"C:\Users\admin\Desktop\bulk_mode_profile.png",
            bbox_inches='tight', pad_inches=0, dpi=600, facecolor='w')
plt.show()
